# 05 · Research loop — connect → pull → upload → compute → save

The full loop the Studio ships to close: connect once, pull platform data
in one line, bring your own data, compute, save/share the result.

**Before running:** set `NYQUIST_API_KEY` (in Studio the per-session
`nyq-use` key is already in the environment — nothing to configure). This
notebook is committed as the demo artifact for the loop; live execution
against the deployed API is deploy-deferred for this phase — the exact
call sequence below is covered end-to-end by the hermetic SDK test suite
against a mocked transport (`test_research_loop_sequence_upload_load_var_save`
in `tests/test_datasets.py`).

In [ ]:
from nyquist import Nyquist
import pandas as pd

nq = Nyquist()  # reads NYQUIST_API_KEY from the env
nq

## 1 · Platform data, one line

`nq.marketdata.history` pulls reconciled OHLCV from the Golden Record
store — no separate auth, no vendor wiring.

In [ ]:
px = nq.marketdata.history("NVDA", days=365)
print(len(px), "bars ·", px.attrs.get("source"))
px.tail()

## 2 · Bring your own data

Upload a DataFrame straight from memory — it round-trips byte-identical.
`nq.datasets.upload` also accepts a local `.csv`/`.parquet` path if you'd
rather read from disk.

In [ ]:
window = px.index[-30:]
my_signals = pd.DataFrame(
    {"conviction": [0.60 + 0.01 * (i % 6) for i in range(len(window))]},
    index=window,  # DatetimeIndex — round-trips through upload/load so the join below lines up
)
ds = nq.datasets.upload(my_signals, name="my_signals")
ds

In [ ]:
back = nq.datasets.load(ds["id"])
back

## 3 · Compute

Join the platform series with your own, then run the risk engine on the
combined return series.

In [ ]:
joined = px[["close"]].join(back, how="right")
returns = joined["close"].pct_change().dropna().tolist()
var_result = nq.risk.var(returns, confidence=0.95)
var_result

## 4 · Save & share

`nq.reports.save` persists an HTML strategy report under your
owner-scoped key. The saved report is visible in the portal under
**Studio → RESEARCH → Reports** (shipped in 41-05) and downloadable via
`nq.reports.download_url`.

In [ ]:
report = nq.reports.save(
    title="My research — NVDA + my_signals",
    returns=returns,
    metrics=var_result,
)
print(report["key"])

In [ ]:
nq.reports.download_url(report["key"])

## What else the SDK reaches

`nq.stress`, `nq.backtest`, `nq.macro`, `nq.crypto`, `nq.agents`,
`nq.ledger` are all one-liners over the same client, plus the generic
`nq.get`/`nq.post` escape-hatch for any endpoint that doesn't have a
dedicated sub-client yet. See the portal's SDK quickstart page for the
full surface.